# Entrega 3 · Modelo predictivo de caída de matrícula por departamento

**Pregunta BA (P3 de la Entrega 1):** ¿Qué departamentos tendrán caída de matrícula el próximo año?

**Problema:** clasificación binaria. Para cada departamento y año *t* se predice si la matrícula del semestre 1 del año *t+1* será menor que la del año *t*.

**Cómo ejecutarlo**
1. Con acceso al data warehouse: defina las variables de entorno `DB_HOST`, `DB_NAME`, `DB_USER` y `DB_PASS` antes de abrir el notebook. El notebook corre `extraccion.sql` y guarda el resultado en `datos/dataset_depto_anio.csv`.
2. Sin acceso al data warehouse (por ejemplo, en Google Colab): el notebook lee `datos/dataset_depto_anio.csv`, que es el resultado de `extraccion.sql` (unas 230 filas).
3. Ejecute *Reiniciar y ejecutar todo*. Los resultados quedan en `resultados/`.

Las contraseñas nunca se escriben en el notebook ni en el repositorio.

In [ ]:
import os, warnings
from pathlib import Path
from urllib.parse import quote_plus
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# El notebook trabaja dentro de la carpeta modelo/
if Path("modelo").is_dir() and not Path("extraccion.sql").exists():
    os.chdir("modelo")

RUTA_SQL = Path("extraccion.sql")
RUTA_DATOS = Path("datos/dataset_depto_anio.csv")
RES = Path("resultados")
FIG = RES / "figuras"
for p in (RUTA_DATOS.parent, RES, FIG):
    p.mkdir(parents=True, exist_ok=True)

SEMILLA = 42
ULTIMO_ANIO_BASE_ENTRENAMIENTO = 2018   # se entrena con años base <= 2018 (objetivo hasta 2019)
RECALL_MINIMO = 0.60                    # criterio de éxito (sección 1.3)
TABLA_PRED = "pred_caida_matricula_depto"

def num(n):      # 1234567 -> 1.234.567
    return f"{int(round(n)):,}".replace(",", ".")
def dec(x, d=2): # 0.5 -> 0,50
    return f"{x:.{d}f}".replace(".", ",")
def pct(x, d=1): # 0.5 -> 50,0 %
    return dec(x * 100, d) + " %"
def md_tabla(df):
    cols = list(df.columns)
    filas = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        filas.append("| " + " | ".join((f"{v:.3f}".replace(".", ",") if isinstance(v, float) else str(v)) for v in r.tolist()) + " |")
    return "\n".join(filas)

MD = []   # líneas del resumen de resultados que se guarda al final

## 2. Comprensión de los datos

### 2.1 Extracción desde el data warehouse

In [ ]:
def crear_engine():
    clave = os.environ.get("DB_PASS")
    if clave is None:
        return None
    usuario = os.environ.get("DB_USER", "etl")
    host = os.environ.get("DB_HOST", "localhost")
    base = os.environ.get("DB_NAME", "matricula")
    return create_engine(f"mysql+pymysql://{usuario}:{quote_plus(clave)}@{host}/{base}?charset=utf8mb4")

engine = crear_engine()
if engine is not None:
    lineas = [l for l in RUTA_SQL.read_text(encoding="utf-8").splitlines()
              if not l.strip().startswith("--") and not l.strip().upper().startswith("USE ")]
    df_raw = pd.read_sql(text("\n".join(lineas)), engine)
    df_raw.to_csv(RUTA_DATOS, index=False)
    ORIGEN = "data warehouse (MySQL), consulta modelo/extraccion.sql"
else:
    if not RUTA_DATOS.exists():
        raise FileNotFoundError(
            "No hay conexión (falta DB_PASS) ni archivo datos/dataset_depto_anio.csv. "
            "Ejecute extraccion.sql en el data warehouse y exporte el resultado a esa ruta.")
    df_raw = pd.read_csv(RUTA_DATOS, dtype={"cod_dpto": str})
    ORIGEN = f"archivo {RUTA_DATOS} (resultado de modelo/extraccion.sql)"

df_raw["cod_dpto"] = df_raw["cod_dpto"].astype(str).str.strip()
print("Origen de los datos:", ORIGEN)
print("Filas:", len(df_raw), "| departamentos:", df_raw.cod_dpto.nunique(),
      "| años:", df_raw.anio.min(), "-", df_raw.anio.max())
df_raw.head()

### 2.2 Descripción del conjunto de datos

In [ ]:
print(df_raw.dtypes)
print("\nNulos por columna:\n", df_raw.isna().sum())
print("\nResumen numérico:")
display(df_raw[["matricula_s1", "matricula_acreditada", "matricula_mujeres", "n_programas", "n_municipios"]].describe().round(1))

tabla_vars = pd.DataFrame([
    ["cod_dpto", "Texto", "Identificador", "dim_geografia.cod_dpto", "Código DANE del departamento"],
    ["nom_dpto", "Texto", "Identificador", "dim_geografia.nom_dpto", "Nombre del departamento"],
    ["anio", "Numérica", "Identificador", "dim_tiempo.anio", "Año base t (semestre 1)"],
    ["matricula_s1", "Numérica", "Entrada (base de derivadas)", "fact_matricula.total_matriculados", "Estudiantes matriculados en el semestre 1 del año t"],
    ["matricula_acreditada", "Numérica", "Entrada (base de derivadas)", "fact_matricula.total_matriculados + dim_institucion.acreditada_alta_calidad", "Matriculados en instituciones con acreditación de alta calidad"],
    ["matricula_mujeres", "Numérica", "Entrada (base de derivadas)", "fact_matricula.total_matriculados + dim_genero.descripcion_genero", "Matriculadas de género femenino"],
    ["n_programas", "Numérica", "Entrada (base de derivadas)", "fact_matricula.sk_programa", "Programas académicos distintos con matrícula"],
    ["n_municipios", "Numérica", "Entrada (base de derivadas)", "fact_matricula.sk_geografia", "Municipios del departamento con matrícula"],
    ["cae_siguiente", "Binaria (0/1)", "Objetivo", "fact_matricula.total_matriculados (años t y t+1)", "1 si la matrícula del semestre 1 de t+1 es menor que la de t"],
    ["log_matricula", "Numérica", "Entrada", "Derivada de matricula_s1", "Logaritmo natural de la matrícula del año t"],
    ["var_anual", "Numérica", "Entrada", "Derivada de matricula_s1 (t y t-1)", "Variación porcentual frente al año anterior"],
    ["cayo_anio_actual", "Binaria (0/1)", "Entrada", "Derivada de var_anual", "1 si la matrícula ya cayó en el año t"],
    ["pct_acreditada", "Numérica", "Entrada", "Derivada de matricula_acreditada", "Proporción de la matrícula en instituciones acreditadas"],
    ["pct_mujeres", "Numérica", "Entrada", "Derivada de matricula_mujeres", "Proporción de la matrícula femenina"],
    ["log_programas", "Numérica", "Entrada", "Derivada de n_programas", "log(1 + programas con matrícula)"],
    ["log_municipios", "Numérica", "Entrada", "Derivada de n_municipios", "log(1 + municipios con matrícula)"],
    ["region", "Categórica", "Entrada", "dim_geografia.cod_dpto + clasificación regional externa", "Región natural del departamento (Andina, Caribe, Pacífica, Orinoquía, Amazonía, Insular)"],
], columns=["Variable", "Tipo", "Papel", "Origen (tabla.columna)", "Descripción"])
tabla_vars

### 2.3 y 2.4 Construcción del panel, variable objetivo y exploración

In [ ]:
# --- Panel departamento-año -------------------------------------------------------
REGION = {  # código DANE del departamento -> región natural
    5: "Andina", 8: "Caribe", 11: "Andina", 13: "Caribe", 15: "Andina", 17: "Andina",
    18: "Amazonía", 19: "Pacífica", 20: "Caribe", 23: "Caribe", 25: "Andina", 27: "Pacífica",
    41: "Andina", 44: "Caribe", 47: "Caribe", 50: "Orinoquía", 52: "Pacífica", 54: "Andina",
    63: "Andina", 66: "Andina", 68: "Andina", 70: "Caribe", 73: "Andina", 76: "Pacífica",
    81: "Orinoquía", 85: "Orinoquía", 86: "Amazonía", 88: "Insular", 91: "Amazonía",
    94: "Amazonía", 95: "Amazonía", 97: "Amazonía", 99: "Orinoquía"}

anios = sorted(df_raw.anio.unique())
n_filas_raw = len(df_raw)

# Departamentos con matrícula en todos los años (necesario para calcular variaciones)
con_datos = df_raw[df_raw.matricula_s1 > 0].groupby("cod_dpto").anio.nunique()
completos = con_datos[con_datos == len(anios)].index
excl = df_raw.loc[~df_raw.cod_dpto.isin(completos)]
nombres_excl = sorted(excl.nom_dpto.unique())

d = df_raw[df_raw.cod_dpto.isin(completos)].sort_values(["cod_dpto", "anio"]).copy()
g = d.groupby("cod_dpto")
d["matricula_anterior"] = g.matricula_s1.shift(1)
d["matricula_siguiente"] = g.matricula_s1.shift(-1)
d["var_anual_original"] = d.matricula_s1 / d.matricula_anterior - 1
d["var_siguiente"] = d.matricula_siguiente / d.matricula_s1 - 1
d["cayo_anio_actual"] = np.where(d.var_anual_original.isna(), np.nan, (d.var_anual_original < 0).astype(float))
d["log_matricula"] = np.log(d.matricula_s1)
d["pct_acreditada"] = d.matricula_acreditada / d.matricula_s1
d["pct_mujeres"] = d.matricula_mujeres / d.matricula_s1
d["log_programas"] = np.log1p(d.n_programas)
d["log_municipios"] = np.log1p(d.n_municipios)
d["cae_siguiente"] = np.where(d.matricula_siguiente.isna(), np.nan, (d.matricula_siguiente < d.matricula_s1).astype(float))
d["region"] = d.cod_dpto.map(lambda c: REGION.get(int(c), "Sin región"))
if (d.region == "Sin región").any():
    print("Aviso: códigos sin región:", d.loc[d.region == "Sin región", "cod_dpto"].unique())

nulos_primer_anio = int(d.var_anual_original.isna().sum())
nulos_ultimo_anio = int(d.cae_siguiente.isna().sum())
ULT_ANIO = int(d.anio.max())

# Filas con variable objetivo conocida (años base 2016 a 2020) y filas para el futuro (año base 2021)
lab = d[d.cae_siguiente.notna() & d.var_anual_original.notna()].copy()
fut = d[(d.anio == ULT_ANIO)].copy()
print(f"Departamentos en el panel: {d.cod_dpto.nunique()} | excluidos por falta de matrícula: {len(nombres_excl)} {nombres_excl}")
print(f"Filas etiquetadas: {len(lab)} | filas para el año futuro: {len(fut)}")
print("Años base etiquetados:", sorted(lab.anio.unique()))

# --- 2.3 Distribución de la variable objetivo ------------------------------------
cuenta_clase = lab.cae_siguiente.value_counts().reindex([0.0, 1.0]).fillna(0).astype(int)
p_cae = cuenta_clase[1.0] / cuenta_clase.sum()
q1, q3 = lab.var_siguiente.quantile([.25, .75])
iqr = q3 - q1
f_inf, f_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
atip_obj = int(((lab.var_siguiente < f_inf) | (lab.var_siguiente > f_sup)).sum())
print(f"Clase 1 (cae): {cuenta_clase[1.0]} ({pct(p_cae)}) | clase 0 (no cae): {cuenta_clase[0.0]}")
print(f"Variación del año siguiente: mín {pct(lab.var_siguiente.min())}, mediana {pct(lab.var_siguiente.median())}, máx {pct(lab.var_siguiente.max())}; atípicos (regla IQR): {atip_obj}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(["No cae (0)", "Cae (1)"], cuenta_clase.values, color=["#4c78a8", "#e45756"])
for i, v in enumerate(cuenta_clase.values):
    ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[0].set_title("Departamentos-año según si la matrícula cae al año siguiente"); ax[0].set_ylabel("Registros")
ax[1].hist(lab.var_siguiente * 100, bins=25, color="#4c78a8", edgecolor="white")
ax[1].axvline(0, color="#e45756", ls="--"); ax[1].set_title("Variación de la matrícula al año siguiente (%)"); ax[1].set_xlabel("%")
plt.tight_layout(); plt.savefig(FIG / "fig_2_3_objetivo.png", bbox_inches="tight"); plt.show()

In [ ]:
# --- 2.4 Gráficos exploratorios ---------------------------------------------------
etq = {0.0: "No cae", 1.0: "Cae"}
def caja(col, titulo, nombre, pct_eje=False):
    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    datos = [lab.loc[lab.cae_siguiente == k, col] * (100 if pct_eje else 1) for k in (0.0, 1.0)]
    ax.boxplot(datos, labels=["No cae al año siguiente", "Cae al año siguiente"], patch_artist=True,
               boxprops=dict(facecolor="#9ecae1"), medianprops=dict(color="#e45756"))
    ax.set_title(titulo)
    plt.tight_layout(); plt.savefig(FIG / nombre, bbox_inches="tight"); plt.show()

caja("var_anual_original", "Variación del año actual (%) según si cae al siguiente", "fig_2_4a_var_anual.png", True)
caja("pct_acreditada", "% de matrícula en instituciones acreditadas (%)", "fig_2_4b_acreditada.png", True)
caja("log_matricula", "Tamaño del departamento (log de la matrícula)", "fig_2_4c_tamano.png")

por_anio = lab.groupby("anio").cae_siguiente.mean() * 100
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.bar((por_anio.index + 1).astype(str), por_anio.values, color="#e45756")
ax.set_title("% de departamentos con caída de matrícula por año"); ax.set_ylabel("% de departamentos")
plt.tight_layout(); plt.savefig(FIG / "fig_2_4d_por_anio.png", bbox_inches="tight"); plt.show()

resumen_eda = lab.groupby("cae_siguiente")[["var_anual_original", "pct_acreditada", "pct_mujeres", "log_matricula", "n_programas"]].median().round(3)
print("Medianas por clase (0 = no cae, 1 = cae):"); display(resumen_eda)
print("Porcentaje de departamentos que caen por año objetivo:"); print(por_anio.round(1).rename(lambda a: a + 1).to_string())

## 3. Preparación de los datos

In [ ]:
# --- 3.1 Nulos y atípicos ---------------------------------------------------------
# Nulos estructurales: el primer año de cada departamento no tiene año anterior; el último no tiene año siguiente.
# Atípicos: se acota var_anual con la regla IQR calculada solo con el conjunto de entrenamiento.
entren_tmp = lab[lab.anio <= ULTIMO_ANIO_BASE_ENTRENAMIENTO]
a1, a3 = entren_tmp.var_anual_original.quantile([.25, .75])
lim_inf, lim_sup = a1 - 1.5 * (a3 - a1), a3 + 1.5 * (a3 - a1)
n_atip_var = int(((d.var_anual_original < lim_inf) | (d.var_anual_original > lim_sup)).sum())
d["var_anual"] = d.var_anual_original.clip(lim_inf, lim_sup)
lab["var_anual"] = lab.var_anual_original.clip(lim_inf, lim_sup)
fut["var_anual"] = fut.var_anual_original.clip(lim_inf, lim_sup)
print(f"Límites de var_anual (IQR, entrenamiento): [{pct(lim_inf)}, {pct(lim_sup)}] | valores acotados: {n_atip_var}")
print("Nulos restantes en las entradas del conjunto etiquetado:", int(lab[["log_matricula","var_anual","cayo_anio_actual","pct_acreditada","pct_mujeres","log_programas","log_municipios","region"]].isna().sum().sum()))

# --- Codificación de la variable categórica (región) -----------------------------
NUM = ["log_matricula", "var_anual", "cayo_anio_actual", "pct_acreditada", "pct_mujeres", "log_programas", "log_municipios"]
regiones = sorted(d.region.unique())
def preparar_X(df_):
    X = df_[NUM].copy()
    for r in regiones:
        X["region_" + r] = (df_.region == r).astype(int)
    return X

# --- 3.3 División temporal --------------------------------------------------------
train = lab[lab.anio <= ULTIMO_ANIO_BASE_ENTRENAMIENTO]
test = lab[lab.anio > ULTIMO_ANIO_BASE_ENTRENAMIENTO]
assert len(train) > 0 and len(test) > 0, "La división temporal dejó un conjunto vacío"
X_train, y_train = preparar_X(train), train.cae_siguiente.astype(int)
X_test, y_test = preparar_X(test), test.cae_siguiente.astype(int)
X_fut = preparar_X(fut)
print(f"Entrenamiento: {len(train)} registros, años base {train.anio.min()}-{train.anio.max()} (objetivo {train.anio.min()+1}-{train.anio.max()+1}), {pct(y_train.mean())} caen")
print(f"Prueba: {len(test)} registros, años base {test.anio.min()}-{test.anio.max()} (objetivo {test.anio.min()+1}-{test.anio.max()+1}), {pct(y_test.mean())} caen")
print("Columnas de entrada:", list(X_train.columns))

## 4. Modelado

In [ ]:
PARAMS_LR = dict(C=1.0, class_weight="balanced", max_iter=1000, random_state=SEMILLA)
PARAMS_RF = dict(n_estimators=300, max_depth=4, min_samples_leaf=3, class_weight="balanced", random_state=SEMILLA)

modelos = {
    "Regresión logística": Pipeline([("escala", StandardScaler()), ("clf", LogisticRegression(**PARAMS_LR))]),
    "Bosque aleatorio": RandomForestClassifier(**PARAMS_RF),
}
for nombre, m in modelos.items():
    m.fit(X_train, y_train)        # mismo conjunto de entrenamiento y mismas variables de entrada
print("Modelos entrenados:", list(modelos))

## 5. Evaluación

In [ ]:
def evaluar(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                exactitud=accuracy_score(y, pred), precision=precision_score(y, pred, zero_division=0),
                recall=recall_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0))

def lectura(nombre, m):
    return (f"De {m['tp'] + m['fn']} departamentos que sí cayeron, {nombre} detectó {m['tp']} y dejó pasar {m['fn']}; "
            f"además marcó como caída a {m['fp']} que no cayeron (de {m['tn'] + m['fp']} que no cayeron).")

resultados = {n: evaluar(y_test, m.predict(X_test)) for n, m in modelos.items()}

# Referencias (no cuentan como algoritmos): clase mayoritaria y "persistencia" (si cayó este año, caerá el siguiente)
base_mayor = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
ref_mayor = evaluar(y_test, base_mayor.predict(X_test))
ref_persist = evaluar(y_test, test.cayo_anio_actual.astype(int))

tabla51 = pd.DataFrame([
    [n, r["tp"] + r["fn"], r["tp"], r["fn"], r["fp"], pct(r["exactitud"]), pct(r["precision"]), pct(r["recall"]), dec(r["f1"])]
    for n, r in {**resultados, "Referencia: siempre «no cae»": ref_mayor, "Referencia: persistencia": ref_persist}.items()],
    columns=["Algoritmo", "Caídas reales", "Detectadas", "No detectadas", "Falsas alarmas", "Exactitud", "Precisión", "Recall (caídas)", "F1"])
display(tabla51)
for n, r in resultados.items():
    print("-", lectura(n, r))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for a, (n, r) in zip(ax, resultados.items()):
    M = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]])
    a.imshow(M, cmap="Blues")
    for (i, j), v in np.ndenumerate(M):
        a.text(j, i, str(v), ha="center", va="center", fontsize=14, color="white" if v > M.max() / 2 else "black")
    a.set_xticks([0, 1]); a.set_xticklabels(["No cae", "Cae"]); a.set_yticks([0, 1]); a.set_yticklabels(["No cae", "Cae"])
    a.set_xlabel("Predicho"); a.set_ylabel("Real"); a.set_title(n); a.grid(False)
plt.tight_layout(); plt.savefig(FIG / "fig_5_1_matrices.png", bbox_inches="tight"); plt.show()

In [ ]:
# --- 5.2 Elección del algoritmo: mayor F1 de la clase "cae"; en empate, mayor recall ----
elegido = max(resultados, key=lambda n: (round(resultados[n]["f1"], 6), resultados[n]["recall"]))
otro = [n for n in resultados if n != elegido][0]
re, ro = resultados[elegido], resultados[otro]
print(f"Algoritmo elegido: {elegido}")
print(f"F1 {dec(re['f1'])} frente a {dec(ro['f1'])} de {otro}; recall {pct(re['recall'])} frente a {pct(ro['recall'])}.")

# Resultado por año de prueba del modelo elegido
modelo_el = modelos[elegido]
filas_anio = []
for a in sorted(test.anio.unique()):
    sel = test.anio == a
    r = evaluar(test.loc[sel, "cae_siguiente"].astype(int), modelo_el.predict(X_test[sel.values]))
    filas_anio.append([int(a) + 1, int(sel.sum()), r["tp"] + r["fn"], r["tp"], r["fp"], pct(r["exactitud"])])
por_anio_test = pd.DataFrame(filas_anio, columns=["Año objetivo", "Departamentos", "Caídas reales", "Detectadas", "Falsas alarmas", "Exactitud"])
display(por_anio_test)

# --- 5.3 Variables más importantes ------------------------------------------------
cols = list(X_train.columns)
if elegido == "Bosque aleatorio":
    imp = pd.Series(modelo_el.feature_importances_, index=cols); tipo_imp = "Importancia (reducción de impureza)"
else:
    imp = pd.Series(modelo_el.named_steps["clf"].coef_[0], index=cols); tipo_imp = "Coeficiente (variables estandarizadas)"
orden = imp.abs().sort_values(ascending=False).index
top = imp.loc[orden].head(8)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index[::-1], top.values[::-1], color="#4c78a8"); ax.set_title(f"Variables más importantes · {elegido}"); ax.set_xlabel(tipo_imp)
plt.tight_layout(); plt.savefig(FIG / "fig_5_3_importancia.png", bbox_inches="tight"); plt.show()
m1 = X_train[y_train == 1].mean(); m0 = X_train[y_train == 0].mean()
tabla_imp = pd.DataFrame({"Variable": top.index, tipo_imp: top.values.round(3),
                          "Promedio si cae": m1.loc[top.index].round(3).values, "Promedio si no cae": m0.loc[top.index].round(3).values})
display(tabla_imp)

# --- 5.4 Criterio de éxito ---------------------------------------------------------
cumple_recall = re["recall"] >= RECALL_MINIMO
cumple_ref = re["exactitud"] > ref_mayor["exactitud"]
tabla54 = pd.DataFrame([
    ["Detectar al menos 6 de cada 10 departamentos que caen (recall ≥ 60 %)", pct(re["recall"]) + f" ({re['tp']} de {re['tp'] + re['fn']})", "Sí" if cumple_recall else "No"],
    ["Exactitud mayor que la de predecir siempre «no cae»", f"{pct(re['exactitud'])} frente a {pct(ref_mayor['exactitud'])}", "Sí" if cumple_ref else "No"],
], columns=["Criterio (sección 1.3)", "Valor obtenido", "¿Se alcanzó?"])
display(tabla54)

## 6. Despliegue

In [ ]:
# --- Respuesta a la pregunta de negocio: departamentos con riesgo de caída en el año siguiente al último ----
def nuevo_modelo(nombre):
    if nombre == "Bosque aleatorio":
        return RandomForestClassifier(**PARAMS_RF)
    return Pipeline([("escala", StandardScaler()), ("clf", LogisticRegression(**PARAMS_LR))])

modelo_final = nuevo_modelo(elegido)
X_lab_all = pd.concat([X_train, X_test]); y_lab_all = pd.concat([y_train, y_test])
modelo_final.fit(X_lab_all, y_lab_all)   # se reentrena con todos los años etiquetados para predecir el futuro

def prob1(m, X):
    return m.predict_proba(X)[:, list(m.classes_).index(1)]

def armar(df_, X_, m, conjunto):
    out = pd.DataFrame({
        "sk_tiempo_base": df_.sk_tiempo.astype(int).values, "cod_dpto": df_.cod_dpto.values,
        "anio_objetivo": (df_.anio + 1).astype(int).values, "conjunto": conjunto, "modelo": elegido,
        "prob_caida": prob1(m, X_).round(4), "caida_predicha": m.predict(X_).astype(int),
        "caida_real": df_.cae_siguiente.values, "matricula_base": df_.matricula_s1.astype(int).values,
        "matricula_real": df_.matricula_siguiente.values})
    out["caida_real"] = out["caida_real"].astype("Int64")
    out["matricula_real"] = out["matricula_real"].round().astype("Int64")
    return out

pred = pd.concat([armar(train, X_train, modelo_el, "entrenamiento"),
                  armar(test, X_test, modelo_el, "prueba"),
                  armar(fut, X_fut, modelo_final, "futuro")], ignore_index=True)
pred = pred.drop_duplicates(["sk_tiempo_base", "cod_dpto"])
pred.to_csv(RES / "predicciones.csv", index=False)

nom = d[["cod_dpto", "nom_dpto"]].drop_duplicates()
riesgo = pred[pred.conjunto == "futuro"].merge(nom, on="cod_dpto").sort_values("prob_caida", ascending=False)
riesgo[["nom_dpto", "prob_caida", "caida_predicha", "matricula_base"]].to_csv(RES / "riesgo_proximo_anio.csv", index=False)
marcados = riesgo[riesgo.caida_predicha == 1]
total_base = riesgo.matricula_base.sum()
print(f"Año objetivo: {int(riesgo.anio_objetivo.iloc[0])} | departamentos con caída predicha: {len(marcados)} de {len(riesgo)}")
print(f"Matrícula 2021 (S1) en esos departamentos: {num(marcados.matricula_base.sum())} de {num(total_base)} ({pct(marcados.matricula_base.sum() / total_base)})")
display(riesgo[["nom_dpto", "prob_caida", "caida_predicha", "matricula_base"]].head(10))

# --- Script SQL de carga (para ejecutarlo en DBeaver cuando no hay conexión directa) ----
def v(x):
    if pd.isna(x): return "NULL"
    if isinstance(x, str): return "'" + x.replace("'", "''") + "'"
    return str(x)
cols_sql = ["sk_tiempo_base", "cod_dpto", "anio_objetivo", "conjunto", "modelo", "prob_caida", "caida_predicha", "caida_real", "matricula_base", "matricula_real"]
lineas = ["USE matricula;", f"DELETE FROM {TABLA_PRED};"]
for r in pred[cols_sql].itertuples(index=False):
    lineas.append(f"INSERT INTO {TABLA_PRED} ({', '.join(cols_sql)}) VALUES ({', '.join(v(x) for x in r)});")
(RES / "carga_predicciones.sql").write_text("\n".join(lineas) + "\n", encoding="utf-8")

# --- Carga directa al data warehouse si hay conexión ------------------------------
if engine is not None:
    try:
        with engine.begin() as con:
            con.execute(text(f"DELETE FROM {TABLA_PRED}"))
            pred[cols_sql].to_sql(TABLA_PRED, con, if_exists="append", index=False)
        print(f"Se cargaron {len(pred)} filas en {TABLA_PRED}")
    except Exception as e:
        print("No se pudo cargar en el DW (¿se ejecutó dw/ddl/07_pred_caida_matricula_depto.sql?):", e)
else:
    print(f"Sin conexión: ejecute resultados/carga_predicciones.sql en el DW ({len(pred)} filas).")

## Resumen de resultados para el documento

In [ ]:
L = MD.append
n_tr, n_te = len(train), len(test)
L("# Resumen de resultados (generado por el notebook)\n")
L("## 2.2 Conjunto de datos")
L(f"- Origen: {ORIGEN}.")
L(f"- Filas extraídas: {num(n_filas_raw)} (departamento-año); periodo {int(min(anios))}-{int(max(anios))}; {d.cod_dpto.nunique()} departamentos en el panel.")
L(f"- Excluidos por no tener matrícula en todos los años: {len(nombres_excl)} {nombres_excl}.")
L(f"- Registros etiquetados: {len(lab)}; filas para predecir el año {ULT_ANIO + 1}: {len(fut)}.\n")
L(md_tabla(tabla_vars) + "\n")
L("## 2.3 Variable objetivo")
L(f"- Cae (1): {cuenta_clase[1.0]} registros ({pct(p_cae)}); no cae (0): {cuenta_clase[0.0]}.")
L(f"- Variación del año siguiente: mínima {pct(lab.var_siguiente.min())}, mediana {pct(lab.var_siguiente.median())}, máxima {pct(lab.var_siguiente.max())}; {atip_obj} atípicos por la regla IQR.\n")
L("## 2.4 Exploración")
L("Medianas por clase (0 = no cae, 1 = cae):\n"); L(md_tabla(resumen_eda.reset_index().round(3)) + "\n")
L("% de departamentos que caen por año objetivo: " + "; ".join(f"{int(a) + 1}: {dec(v_, 1)} %" for a, v_ in por_anio.items()) + "\n")
L("## 3.1 Limpieza y transformación")
L(md_tabla(pd.DataFrame([
    ["Nulos estructurales", "var_anual, cayo_anio_actual", "Sin año anterior en el primer año de cada departamento: se descartan para el modelo", nulos_primer_anio],
    ["Nulos estructurales", "cae_siguiente", "Sin año siguiente en el último año: esas filas se usan solo para predecir el futuro", nulos_ultimo_anio],
    ["Departamentos incompletos", "todas", "Se excluyen los departamentos sin matrícula en todos los años", int((~df_raw.cod_dpto.isin(completos)).sum())],
    ["Atípicos", "var_anual", f"Se acota a [{pct(lim_inf)}, {pct(lim_sup)}] (regla IQR con el entrenamiento)", n_atip_var],
    ["Transformación", "matricula_s1, n_programas, n_municipios", "Logaritmo para reducir la asimetría", "—"],
    ["Codificación categórica", "region", "One-hot encoding (una columna 0/1 por región)", "—"]],
    columns=["Tratamiento", "Variable(s)", "Qué se hizo", "Registros afectados"])) + "\n")
L("## 3.3 División")
L(md_tabla(pd.DataFrame([
    ["Entrenamiento", num(n_tr), f"Años base {train.anio.min()}-{train.anio.max()} (objetivo {train.anio.min() + 1}-{train.anio.max() + 1})"],
    ["Prueba", num(n_te), f"Años base {test.anio.min()}-{test.anio.max()} (objetivo {test.anio.min() + 1}-{test.anio.max() + 1})"]],
    columns=["Conjunto", "Registros", "Periodo"])) + "\n")
L(f"En entrenamiento caen {pct(y_train.mean())} de los casos; en prueba, {pct(y_test.mean())}.\n")
L("## 4.1 Algoritmos")
L(f"- Regresión logística: {PARAMS_LR}, variables estandarizadas.")
L(f"- Bosque aleatorio: {PARAMS_RF}.\n")
L("## 5.1 Métricas sobre el conjunto de prueba")
L(md_tabla(tabla51) + "\n")
for n, r in resultados.items():
    L("- " + lectura(n, r))
L("\n## 5.2 Elección")
L(f"Se elige **{elegido}**: F1 de la clase «cae» de {dec(re['f1'])} frente a {dec(ro['f1'])} de {otro}; recall de {pct(re['recall'])} frente a {pct(ro['recall'])}; exactitud de {pct(re['exactitud'])} frente a {pct(ro['exactitud'])}.\n")
L(md_tabla(por_anio_test) + "\n")
L("## 5.3 Variables más importantes\n"); L(md_tabla(tabla_imp) + "\n")
L("## 5.4 Criterio de éxito\n"); L(md_tabla(tabla54) + "\n")
L("## 5.5 Respuesta a la pregunta de negocio")
L(f"Para {ULT_ANIO + 1} el modelo marca {len(marcados)} de {len(riesgo)} departamentos con caída de matrícula; concentran {num(marcados.matricula_base.sum())} estudiantes ({pct(marcados.matricula_base.sum() / total_base)} de la matrícula del semestre 1 de {ULT_ANIO} en el panel).")
L("Con mayor probabilidad de caída: " + "; ".join(f"{r.nom_dpto} ({pct(r.prob_caida, 0)})" for r in riesgo.head(8).itertuples()) + ".\n")
L("## 6.1 Tabla de predicciones")
L(f"- {TABLA_PRED}: {len(pred)} filas ({(pred.conjunto == 'entrenamiento').sum()} entrenamiento, {(pred.conjunto == 'prueba').sum()} prueba, {(pred.conjunto == 'futuro').sum()} futuro).\n")
L("## Limitaciones a declarar")
L("- Muestra pequeña: un registro por departamento y año; las métricas de prueba se mueven mucho con pocos casos.")
L("- La prueba incluye 2020 (pandemia), con un comportamiento distinto del de entrenamiento.")
L("- La acreditación es la vigente en el catálogo de F1 (fecha de descarga) y se aplica a todos los años.")
(RES / "resumen_resultados.md").write_text("\n".join(MD), encoding="utf-8")
print("\n".join(MD))